# SAA Final Project

The notebook builts portfolios based on JP Morgan Long-Term Capital Market Assumptions - LTCM, using:


1.   Expected Returns for Asset Classes $\mu_i$
2.   Expected Volatility $\sigma_i$
3.   Correlation and Covariance Matrices $\rho_{ij}, \Sigma$

Portfolios are constructed using the library `Skfolio`.


## Libraries and Dependencies

Run including the`pip install skfolio` cell.

In [4]:
#Libraries and dependencies
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

from google.colab import auth
auth.authenticate_user()

import os
import gspread
from google.auth import compute_engine
from google.auth import default

creds, _ = default()
gc = gspread.authorize(creds)

pd.options.display.float_format = '{:.4f}'.format

import sys
import importlib

module_path = '/content/drive/MyDrive/FRE-6921 Final Project/Code'
if module_path not in sys.path:
    sys.path.append(module_path)

import helper
from helper import plot_pair_correlation
importlib.reload(helper)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


ImportError: cannot import name 'plot_pair_correlation' from 'helper' (unknown location)

In [ ]:
#Install Skfolio
%%capture
!pip install skfolio

In [ ]:
from skfolio.optimization import (
    HierarchicalRiskParity,
    MeanRisk,
    SchurComplementary,
)
import fix_prior
from fix_prior import FixPrior

importlib.reload(fix_prior)

## Data

Read the Google Sheets file with the consolidated information ($\mu_i, \Sigma, \rho_{ij}$) for each year.

In [ ]:
# Read LTCM assumptions data
data_path = "/content/drive/MyDrive/FRE-6921 Final Project/Data/LTCM_JPMORGAN_AllMatrix"
file_name = os.path.basename(data_path)
sh = gc.open(file_name)

worksheet_list = sh.worksheets()
tab_count = len(worksheet_list)
print(f"Total number of tabs: {tab_count}")
print(f"Tabs: {worksheet_list}")
worksheet_list_LTCM = worksheet_list[3:]
print(f"Tabs with LTCM assumptions: {worksheet_list_LTCM}")

In [ ]:
# Read return indexes data
col_names = worksheet_list[0].get('A1:O1')
data_range = worksheet_list[0].get('A2:O106', value_render_option='UNFORMATTED_VALUE')
returns_data = pd.DataFrame(data_range, columns = col_names)
returns_data = returns_data.replace(" ", np.nan)
returns_data = returns_data.replace('', np.nan)
returns_data.apply(pd.to_numeric)
returns_data.tail()
#returns_data['Date'] = pd.to_datetime(returns_data['Date'])
#returns_data.reset_index(drop=True)
#returns_data.set_index('Date', inplace=False)
#returns_data.index = pd.to_datetime(returns_data.index)


In [ ]:
returns_dict = {}
vol_dict = {}
correlation_matrices = {}
covariance_matrices = {}

for sheet in worksheet_list_LTCM:
  sheet_name = sheet.title
  # --- Part 1: Expected Returns & Volatility ---
  # Fetching A3:C16 (encompasses asset names, returns, and vol)
  data_range = sheet.get('A3:C16', value_render_option='UNFORMATTED_VALUE')
  temp_df = pd.DataFrame(data_range, columns=['Asset', 'Returns', 'Vol'])

  # Store Asset names as index for the first sheet
  if not returns_dict:
      returns_dict['Asset'] = temp_df['Asset'].values
      vol_dict['Asset'] = temp_df['Asset'].values

  # Append the values using the sheet name as the column header
  returns_dict[sheet_name] = temp_df['Returns'].values
  vol_dict[sheet_name] = temp_df['Vol'].values

  # --- Part 2: Correlation Matrix ---
  # Fetching A3:A16 (Names) and D3:Q16 (Correlation values)
  corr_names = sheet.get('A3:A16')
  corr_values = sheet.get('D3:Q16', value_render_option='UNFORMATTED_VALUE')

  corr_df = pd.DataFrame(corr_values,
                          index=[n[0] for n in corr_names],
                          columns=[n[0] for n in corr_names])
  # Ensure data is float
  corr_df = corr_df.apply(pd.to_numeric)

  # Store in dictionary with sheet name as key
  correlation_matrices[sheet_name] = corr_df

  # --- Part 3: Covariance Matrix ---
  # Fetching A3:A16 (Names) and D2:N16 (Correlation values)

  cov_names = sheet.get('A3:A16')
  cov_values = sheet.get('R3:AE16', value_render_option='UNFORMATTED_VALUE')

  cov_df = pd.DataFrame(cov_values,
                          index=[n[0] for n in cov_names],
                          columns=[n[0] for n in cov_names])

  # Ensure data is float
  cov_df = cov_df.apply(pd.to_numeric)

  # Store in dictionary with sheet name as key
  covariance_matrices[sheet_name] = cov_df

expected_return = pd.DataFrame(returns_dict).set_index('Asset')
expected_vol = pd.DataFrame(vol_dict).set_index('Asset')

### Visualizations

Check that updates of LTCM make sense. Since there are 14 assets and 91 pair correlations, we use helper function to vizualize pairwise correlation for two assets across time.

In [ ]:
expected_return*100

In [ ]:
expected_vol*100

In [ ]:
corr_threshold = 0.2
corr_change = correlation_matrices['2026'] - correlation_matrices['2016']
corr_change = corr_change[corr_change.abs()> corr_threshold]
corr_change

In [ ]:
asset_a = 'U.S. Large Cap'
asset_b = 'U.S. Long Treasuries'
asset_c = 'TIPS'
asset_d = 'U.S. REITS'
asset_e = 'Private Equity'
asset_f = 'U.S. Small Cap'

fig, ax = plt.subplots(figsize=(12, 6))
plot_pair_correlation(correlation_matrices, asset_a, asset_b, ax=ax)
plot_pair_correlation(correlation_matrices, asset_c, asset_d, ax=ax)
plot_pair_correlation(correlation_matrices, asset_e, asset_f, ax=ax)
plt.title('Comparison of Asset Pair Correlations Over Time')
plt.show()

## Portfolios

Implementation of portfolios using `skfolio` and `FixPrior` to pass JP Morgan LTCM assumptions as priors of $\mu, \Sigma, \rho_{ij}$

In [ ]:
#Get return matrix
returns = returns_data.copy()
returns = returns.iloc[:,1:].dropna()
print("Quarterly Return Matrix Dimension: ", returns.shape)

### Schur Complementary Portfolio

This is an example of the implementation of `SchurComplementary` portfolios with LTCM assumptions passed via `FixPrior`, and `gamma = 0.5` without any weight restrictions.

LTCM year/version is chosen as a parameter.

In [ ]:
evaluated_year = '2026'

mu = np.array(expected_return[evaluated_year])
cov_matrix = covariance_matrices[evaluated_year]
target_corr = correlation_matrices[evaluated_year]

LTCM_prior_2016 = FixPrior(mu = mu.T, covariance = cov_matrix, correlation=target_corr, returns = returns)

schur = SchurComplementary(gamma=0.9, prior_estimator=LTCM_prior_2016)
schur

In [ ]:
schur.fit(returns)
schur_weights = pd.Series(schur.weights_, index=returns.columns, name="Schur")
print("-"*50)
print("Optimized Portfolio Weights:")
print(schur_weights)
print("-"*50)

#schur.get_params() #Checked that the prior ReturnDistribution has attributes mu, cov, and corr as passed via FixPrior.

### Hierarchical Risk-Parity and Minimum Variance

With the 2016 LTCM and `FixPrior` class, implement HRP and Mean Risk portfolios and compare weights.

In [ ]:
hrp = HierarchicalRiskParity(prior_estimator=LTCM_prior_2016, portfolio_params={"tag": "HRP"})
hrp.fit(returns)
hrp_weights = pd.Series(hrp.weights_, index=returns.columns, name="HRP")

mean_variance = MeanRisk(
    prior_estimator=LTCM_prior_2016,
    efficient_frontier_size=None,
    max_standard_deviation=None,
    portfolio_params={"tag": "MVO"},
)
mean_variance.fit(returns)
mean_variance_weights = pd.Series(mean_variance.weights_, index=returns.columns, name="MVO")


In [ ]:
#With 2016 LTCM
weights_df = pd.concat([schur_weights, hrp_weights, mean_variance_weights], axis=1)
weights_df

In [ ]:
#Runned with 2026
weights_df = pd.concat([schur_weights, hrp_weights, mean_variance_weights], axis=1)
weights_df